In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

from  lora_transfer_pruning.core.pruning_instrumentor import PruningInstrumentor
from transformers import AutoModelForCausalLM
import torch
import compare_utils
from compare_utils import debug_group_prune_step_by_step
from lora_transfer_pruning.adapter.torch_pruning_group_builder import TorchPruningGroupBuilder
from compare_utils import full_attention_test_with_prune
from datetime import datetime


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [2]:
MODEL = "meta-llama/Llama-3.1-8B-Instruct" 
DEVICE = "cuda:3"
def load_model(device=DEVICE):
    model = AutoModelForCausalLM.from_pretrained(
        MODEL,
        #quantization_config=quantization_config,
        dtype=torch.bfloat16,
        device_map=device,
        #device_map="balanced",
        #max_memory={0: "16GiB", 3: "36GiB"},
    
        # cache_dir="/glazkov-dev/.cache",
    )
    return model

In [3]:
model = load_model()
model

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_fn): Func()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): Llama

In [4]:
# model.model.layers.__len__() #32
NUM_MODEL_LAYERS = 32

In [5]:
def get_batch_size_mb(batch_encoding: dict):
    """Получить размер BatchEncoding в МБ (суммирует все тензоры)
    Args:
        batch_encoding: состоит из input_ids, attention_mask, labels
    """
    total_bytes = sum(
        v.numel() * v.element_size() 
        for v in batch_encoding.values() 
        if torch.is_tensor(v)
    )
    return total_bytes / (1024 ** 2)

def get_model_size_mb(model):
    """Подсчитать примерный размер весов модели в МБ"""
    total_params = 0
    total_bytes = 0
    
    for param in model.parameters():
        num_params = param.numel()
        param_bytes = num_params * param.element_size()
        total_params += num_params
        total_bytes += param_bytes
    
    size_mb = total_bytes / (1024 ** 2)
    
    return size_mb

In [6]:
# get_model_size_mb(model) #15316 mb

In [ ]:
from transformer_lens.model_bridge.bridge import TransformerBridge


# bridge = TransformerBridge.boot_transformers(
#     MODEL,
#     hf_model=model,
#     dtype=torch.float16,
# )

In [8]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
validation_dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="validation",
)
validation_dataset

Dataset({
    features: ['text'],
    num_rows: 3760
})

In [9]:
for i, text in enumerate(validation_dataset):
    print(f"{i}: {text}")
    if i > 10:
        break

0: {'text': ''}
1: {'text': ' = Homarus gammarus = \n'}
2: {'text': ''}
3: {'text': ' Homarus gammarus , known as the European lobster or common lobster , is a species of clawed lobster from the eastern Atlantic Ocean , Mediterranean Sea and parts of the Black Sea . It is closely related to the American lobster , H. americanus . It may grow to a length of 60 cm ( 24 in ) and a mass of 6 kilograms ( 13 lb ) , and bears a conspicuous pair of claws . In life , the lobsters are blue , only becoming " lobster red " on cooking . Mating occurs in the summer , producing eggs which are carried by the females for up to a year before hatching into planktonic larvae . Homarus gammarus is a highly esteemed food , and is widely caught using lobster pots , mostly around the British Isles . \n'}
4: {'text': ''}
5: {'text': ' = = Description = = \n'}
6: {'text': ''}
7: {'text': ' Homarus gammarus is a large crustacean , with a body length up to 60 centimetres ( 24 in ) and weighing up to 5 – 6 kilogram

In [10]:
CONTEXT_LENGTH = 256
NUM_EVAL_BLOCKS = 256 #(block=batch)
EVAL_BATCH_SIZE = 1

validation_text = "\n\n".join(
    text for text in validation_dataset["text"] if text.strip()
)
validation_tokens = tokenizer(
    validation_text,
    add_special_tokens=False,
    return_tensors="pt",
).input_ids[0]

num_blocks = NUM_EVAL_BLOCKS
assert num_blocks > 0, "Validation split does not contain enough tokens"
evaluation_blocks = validation_tokens[: num_blocks * CONTEXT_LENGTH].reshape(
    num_blocks, CONTEXT_LENGTH
)
evaluation_blocks.shape

torch.Size([256, 256])

### Without MOE block, just mlp

### Learning with TP and ours

In [11]:
from prune_and_train_model_for_comparsion import full_load_prune_learn_pipeline
FRACTION_ATTN_LAYERS = [28, 29, 30] #32, 40]
FRACTION_MLP_LAYERS = [28, 29, 30] #30] 
FROZE_N_LAYERS = 28
#ATTN_OUT_FRACTION = 0.1
#MLP_OUT_FRACTION = 0.3 
ATTN_OUT_FRACTION = 0.3#[1, 2, 99, 125] #157, 250] #q proj < 128!!!
MLP_OUT_FRACTION = [3, 5, 1000, 2000, 3001, 4002] #< 4096! #5003, 6004, 7005, 8006, 9007, 10009] #up proj
FRACTION_SEED = 0

In [12]:
optimizer_factory = lambda params: torch.optim.SGD( #works better than AdamW consuming lots of memory
    params,
    lr=1e-4, #1e-3 too much
    momentum=0.0,
    weight_decay=0.0,
    # nesterov=True
)

In [13]:
NUM_STEPS = 500
PRINT_EVERY = 25
BATCH_SIZE = 6

In [14]:
# torch.cuda.memory._record_memory_history()


our_history = full_load_prune_learn_pipeline(MODEL,
                               load_model,
                               evaluation_blocks[:BATCH_SIZE], #will learn 1 batch every time
                               
    FRACTION_ATTN_LAYERS, FRACTION_MLP_LAYERS, ATTN_OUT_FRACTION, MLP_OUT_FRACTION,
    is_torch_pruning=False,
    rescale=False,
    seed=FRACTION_SEED,
    batch_size=BATCH_SIZE,
    optimizer_factory=optimizer_factory,
    num_steps=NUM_STEPS,
    print_every=PRINT_EVERY,
    froze_n_layers=FROZE_N_LAYERS
)

# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# torch.cuda.memory._dump_snapshot(f"llama_prune_our_bs{BATCH_SIZE}x256_{timestamp}.pickle")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

fraction prune_task:
  blocks.28.attn.q_proj: GroupPruneTask(cols=None, rows=0.3, kv_lora_idxs_deepseek=None)
  blocks.29.attn.q_proj: GroupPruneTask(cols=None, rows=0.3, kv_lora_idxs_deepseek=None)
  blocks.30.attn.q_proj: GroupPruneTask(cols=None, rows=0.3, kv_lora_idxs_deepseek=None)
  blocks.28.mlp.up_proj: GroupPruneTask(cols=None, rows=[3, 5, 1000, 2000, 3001, 4002], kv_lora_idxs_deepseek=None)
  blocks.29.mlp.up_proj: GroupPruneTask(cols=None, rows=[3, 5, 1000, 2000, 3001, 4002], kv_lora_idxs_deepseek=None)
  blocks.30.mlp.up_proj: GroupPruneTask(cols=None, rows=[3, 5, 1000, 2000, 3001, 4002], kv_lora_idxs_deepseek=None)


/glazkov-dev/LoRa-Transfer-Pruning/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.layers.22._original_component.post_attention_layernorm._original_component.weight', 'model.layers.27._original_component.post_attention_layernorm._original_component.weight', 'model.layers.3._original_component.post_attention_layernorm._original_component.weight', 'model.layers.6._original_component.post_attention_layernorm._original_component.weight', 'model.layers.25._original_component.post_attention_layernorm._original_component.weight', 'model.layers.0._original_component.input_layernorm._original_component.weight', 'model.layers.1._original_component.input_layernorm._original_component.weight', 'model.layers.28._original_component.post_attention_layernorm._original_component.weight', 'model.layers.30._original_component.input_layernorm._original_component.weight', 'model.layers.9._original_component.input_layernorm._origi

removed group indices by module:
  blocks.28.attn.q_proj: count=1216, idxs=[0, 1, 3, 5, 6, 7, 8, 9, 14, 18, 19, 21, 27, 28, 34, 44, 47, 49, 59, 64, 65, 67, 69, 70, 71, 72, 73, 78, 82, 83, 85, 91, 92, 98, 108, 111, 113, 123, 128, 129, 131, 133, 134, 135, 136, 137, 142, 146, 147, 149, 155, 156, 162, 172, 175, 177, 187, 192, 193, 195, 197, 198, 199, 200, 201, 206, 210, 211, 213, 219, 220, 226, 236, 239, 241, 251, 256, 257, 259, 261, 262, 263, 264, 265, 270, 274, 275, 277, 283, 284, 290, 300, 303, 305, 315, 320, 321, 323, 325, 326, 327, 328, 329, 334, 338, 339, 341, 347, 348, 354, 364, 367, 369, 379, 384, 385, 387, 389, 390, 391, 392, 393, 398, 402, 403, 405, 411, 412, 418, 428, 431, 433, 443, 448, 449, 451, 453, 454, 455, 456, 457, 462, 466, 467, 469, 475, 476, 482, 492, 495, 497, 507, 512, 513, 515, 517, 518, 519, 520, 521, 526, 530, 531, 533, 539, 540, 546, 556, 559, 561, 571, 576, 577, 579, 581, 582, 583, 584, 585, 590, 594, 595, 597, 603, 604, 610, 620, 623, 625, 635, 640, 641, 643, 6

In [17]:
torch.cuda.empty_cache()

In [17]:
our_history

In [33]:
torch.cuda.max_memory_allocated(DEVICE) / 1024 / 1024

33390.9775390625

In [30]:
torch.cuda.max_memory_reserved(DEVICE) / 1024 / 1024

39246.0

In [31]:
torch.cuda.memory_reserved(DEVICE) / 1024 / 1024

39086.0

In [32]:
torch.cuda.memory_allocated(DEVICE) / 1024 / 1024

17.25

In [ ]:
# torch.cuda.memory._record_memory_history()


tp_history = full_load_prune_learn_pipeline(MODEL,
                               load_model,
                               evaluation_blocks[:BATCH_SIZE], #will learn 1 batch every time
                               
    FRACTION_ATTN_LAYERS, FRACTION_MLP_LAYERS, ATTN_OUT_FRACTION, MLP_OUT_FRACTION,
    is_torch_pruning=True,
    #rescale=False,
    seed=FRACTION_SEED,
    batch_size=BATCH_SIZE,
    optimizer_factory=optimizer_factory,
    num_steps=NUM_STEPS,
    print_every=PRINT_EVERY,
    froze_n_layers=FROZE_N_LAYERS
)


# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# torch.cuda.memory._dump_snapshot(f"llama_prune_tp_bs2x256_{timestamp}.pickle")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

fraction prune_task:
  blocks.28.attn.q_proj: GroupPruneTask(cols=None, rows=0.3, kv_lora_idxs_deepseek=None)
  blocks.29.attn.q_proj: GroupPruneTask(cols=None, rows=0.3, kv_lora_idxs_deepseek=None)
  blocks.30.attn.q_proj: GroupPruneTask(cols=None, rows=0.3, kv_lora_idxs_deepseek=None)
  blocks.28.mlp.up_proj: GroupPruneTask(cols=None, rows=[3, 5, 1000, 2000, 3001, 4002], kv_lora_idxs_deepseek=None)
  blocks.29.mlp.up_proj: GroupPruneTask(cols=None, rows=[3, 5, 1000, 2000, 3001, 4002], kv_lora_idxs_deepseek=None)
  blocks.30.mlp.up_proj: GroupPruneTask(cols=None, rows=[3, 5, 1000, 2000, 3001, 4002], kv_lora_idxs_deepseek=None)


/glazkov-dev/LoRa-Transfer-Pruning/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.layers.4._original_component.input_layernorm._original_component.weight', 'model.layers.4._original_component.post_attention_layernorm._original_component.weight', 'model.layers.9._original_component.post_attention_layernorm._original_component.weight', 'model.layers.11._original_component.input_layernorm._original_component.weight', 'model.layers.11._original_component.post_attention_layernorm._original_component.weight', 'model.layers.16._original_component.input_layernorm._original_component.weight', 'model.layers.16._original_component.post_attention_layernorm._original_component.weight', 'model.layers.0._original_component.input_layernorm._original_component.weight', 'model.layers.21._original_component.input_layernorm._original_component.weight', 'model.layers.26._original_component.input_layernorm._original_component.wei

removed group indices by module:
  blocks.28.attn.q_proj: count=1216, idxs=[0, 1, 3, 5, 6, 7, 8, 9, 14, 18, 19, 21, 27, 28, 34, 44, 47, 49, 59, 64, 65, 67, 69, 70, 71, 72, 73, 78, 82, 83, 85, 91, 92, 98, 108, 111, 113, 123, 128, 129, 131, 133, 134, 135, 136, 137, 142, 146, 147, 149, 155, 156, 162, 172, 175, 177, 187, 192, 193, 195, 197, 198, 199, 200, 201, 206, 210, 211, 213, 219, 220, 226, 236, 239, 241, 251, 256, 257, 259, 261, 262, 263, 264, 265, 270, 274, 275, 277, 283, 284, 290, 300, 303, 305, 315, 320, 321, 323, 325, 326, 327, 328, 329, 334, 338, 339, 341, 347, 348, 354, 364, 367, 369, 379, 384, 385, 387, 389, 390, 391, 392, 393, 398, 402, 403, 405, 411, 412, 418, 428, 431, 433, 443, 448, 449, 451, 453, 454, 455, 456, 457, 462, 466, 467, 469, 475, 476, 482, 492, 495, 497, 507, 512, 513, 515, 517, 518, 519, 520, 521, 526, 530, 531, 533, 539, 540, 546, 556, 559, 561, 571, 576, 577, 579, 581, 582, 583, 584, 585, 590, 594, 595, 597, 603, 604, 610, 620, 623, 625, 635, 640, 641, 643, 6

In [1]:
def plot_training_history(history, *, title="Training dynamics"):
    """Plot every numeric metric from a training history in one figure."""
    import math
    import numbers
    import matplotlib.pyplot as plt

    if not history:
        raise ValueError("history is empty")

    steps = [row.get("step", index + 1) for index, row in enumerate(history)]
    metric_names = [
        key for key in dict.fromkeys(key for row in history for key in row)
        if key != "step" and any(isinstance(row.get(key), numbers.Number) for row in history)
    ]
    if not metric_names:
        raise ValueError("history contains no numeric metrics")

    labels = {
        "loss": "Loss", "perplexity": "Perplexity",
        "gradient_norm": "Gradient norm", "learning_rate": "Learning rate",
    }
    ncols = min(2, len(metric_names))
    nrows = math.ceil(len(metric_names) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 4 * nrows),
                             squeeze=False, sharex=True)
    flat_axes = list(axes.flat)

    for ax, metric_name in zip(flat_axes, metric_names):
        values = [row.get(metric_name, float("nan")) for row in history]
        label = labels.get(metric_name, metric_name.replace("_", " ").title())
        ax.plot(steps, values, marker="o", linewidth=2, label=label)
        ax.set(title=label, xlabel="Training step", ylabel=label)
        ax.grid(True, alpha=0.3)
        ax.legend()
        if metric_name == "learning_rate":
            ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))

    for ax in flat_axes[len(metric_names):]:
        ax.set_visible(False)

    fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    return fig, axes


In [2]:
_ = plot_training_history(our_history)

NameError: name 'our_history' is not defined

In [3]:
_ = plot_training_history(tp_history)

NameError: name 'tp_history' is not defined

In [ ]:
from transformers import TrainingArguments

Backlog:

with frozed first 28 layers, bs=3 and cont_len=256 and attn_fraction=0.3 and some channels off in layers 28, 29, 30 I tried to finetune model on one batch (3x256) on two prunings (our and tp). Loss before pruning: 2.10, pp=8.4. After 500 steps: 2.10 and 8.4 for transfer pruning and 2.12 and 8.37 for torch pruning.